In [1]:
import pandas as pd
from pathlib import Path
from src.feature_engineering import (
    extract_date_features,
    create_amenity_count,
    create_amenity_features,
    create_description_features,
    create_host_response_feature,
    create_occupancy_ratios,
    group_property_types,
)

DF_PATH = Path("../data/processed/preprocessed_df.parquet")
df = pd.read_parquet(DF_PATH)

In [2]:
# look at the processed DataFrame
df.head()

,property_type,room_type,amenities,accommodates,bathrooms,bed_type,cancellation_policy,cleaning_fee,city,description,...,last_review,latitude,longitude,neighbourhood,number_of_reviews,review_scores_rating,zipcode,bedrooms,beds,log_price
0,Apartment,Entire home/apt,"{""Wireless Internet"",""Air conditioning"",Kitche...",3,1.0,Real Bed,strict,1,NYC,"Beautiful, sunlit brownstone 1-bedroom in the ...",...,2016-07-18,40.696524,-73.991617,Brooklyn Heights,2,100.0,11201,1.0,1.0,5.010635
1,Apartment,Entire home/apt,"{""Wireless Internet"",""Air conditioning"",Kitche...",7,1.0,Real Bed,strict,1,NYC,Enjoy travelling during your stay in Manhattan...,...,2017-09-23,40.766115,-73.989040,Hell's Kitchen,6,93.0,10019,3.0,3.0,5.129899
2,Apartment,Entire home/apt,"{TV,""Cable TV"",""Wireless Internet"",""Air condit...",5,1.0,Real Bed,moderate,1,NYC,The Oasis comes complete with a full backyard ...,...,2017-09-14,40.808110,-73.943756,Harlem,10,92.0,10027,1.0,3.0,4.976734
3,House,Entire home/apt,"{TV,""Cable TV"",Internet,""Wireless Internet"",Ki...",4,1.0,Real Bed,flexible,1,SF,This light-filled home-away-from-home is super...,...,NaT,37.772004,-122.431619,Lower Haight,0,96.0,94117.0,2.0,2.0,6.620073
4,Apartment,Entire home/apt,"{TV,Internet,""Wireless Internet"",""Air conditio...",2,1.0,Real Bed,moderate,1,DC,"Cool, cozy, and comfortable studio located in ...",...,2017-01-22,38.925627,-77.034596,Columbia Heights,4,40.0,20009,0.0,1.0,4.744932


## Date Features

In [3]:
df = extract_date_features(df)

## Amenity-related Features

In [4]:
# amenity count
df = create_amenity_count(df)

# binary features
df = create_amenity_features(df)

## Description Features

In [5]:
df = create_description_features(df)

## Host Response Rate Binary Column

In [6]:
df = create_host_response_feature(df)

## Occupancy-related Ratios

In [7]:
df = create_occupancy_ratios(df)

## Group rare property types into 'Other'

In [8]:
df = group_property_types(df)

## Check the final DataFrame

In [9]:
# look at the final feature engineered DataFrame
df.head()

,property_type,room_type,accommodates,bathrooms,bed_type,cancellation_policy,cleaning_fee,city,host_has_profile_pic,host_identity_verified,...,amenities_count,has_wifi,has_kitchen,has_heating,description_length,description_word_count,does_host_respond,accommodates_per_bedroom,beds_per_bedroom,bathrooms_per_bedroom
0,Apartment,Entire home/apt,3,1.0,Real Bed,strict,1,NYC,t,t,...,9,1,1,1,211,31,0,3.000000,1.0,1.000000
1,Apartment,Entire home/apt,7,1.0,Real Bed,strict,1,NYC,t,f,...,15,1,1,1,1000,172,1,2.333333,1.0,0.333333
2,Apartment,Entire home/apt,5,1.0,Real Bed,moderate,1,NYC,t,t,...,19,1,1,1,1000,172,1,5.000000,3.0,1.000000
3,House,Entire home/apt,4,1.0,Real Bed,flexible,1,SF,t,t,...,15,1,1,1,468,78,0,2.000000,1.0,0.500000
4,Apartment,Entire home/apt,2,1.0,Real Bed,moderate,1,DC,t,t,...,12,1,1,1,699,120,1,0.000000,0.0,0.000000


In [10]:
# shape
df.shape

(74111, 35)

In [11]:
# verify NaN values
df.isna().sum()

property_type                   0
room_type                       0
accommodates                    0
bathrooms                       0
bed_type                        0
cancellation_policy             0
cleaning_fee                    0
city                            0
host_has_profile_pic            0
host_identity_verified          0
host_response_rate          18299
instant_bookable                0
latitude                        0
longitude                       0
neighbourhood                   0
number_of_reviews               0
review_scores_rating            0
zipcode                         0
bedrooms                        0
beds                            0
log_price                       0
host_year                     188
host_month                    188
host_days                     188
days_since_last_review      15827
amenities_count                 0
has_wifi                        0
has_kitchen                     0
has_heating                     0
description_le

In [12]:
# info
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 74111 entries, 0 to 74110
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   property_type             74111 non-null  str    
 1   room_type                 74111 non-null  str    
 2   accommodates              74111 non-null  int64  
 3   bathrooms                 74111 non-null  float64
 4   bed_type                  74111 non-null  str    
 5   cancellation_policy       74111 non-null  str    
 6   cleaning_fee              74111 non-null  int8   
 7   city                      74111 non-null  str    
 8   host_has_profile_pic      74111 non-null  str    
 9   host_identity_verified    74111 non-null  str    
 10  host_response_rate        55812 non-null  float64
 11  instant_bookable          74111 non-null  str    
 12  latitude                  74111 non-null  float64
 13  longitude                 74111 non-null  float64
 14  neighbourhood    

In [13]:
# move target (log_price) to the end
col_to_move = df.pop("log_price")
df.insert(len(df.columns), "log_price", col_to_move)

In [14]:
# save the feature engineered DataFrame
df.to_parquet("../data/processed/feature_engineered_df.parquet", index=False)